In [3]:
import pandas as pd
import duckdb

# Load the intermediate cleaned dataset created on Day 4.
matches = pd.read_csv("../data/processed/clean_partial.csv")

print("Dataset shape:", matches.shape)
matches.head()

Dataset shape: (836, 20)


,Year,Datetime,Stage,Stadium,City,Home Team Name,Home Team Goals,Away Team Goals,Away Team Name,Win conditions,Attendance,Half-time Home Goals,Half-time Away Goals,Referee,Assistant 1,Assistant 2,RoundID,MatchID,Home Team Initials,Away Team Initials
0,1930.0,1930-07-13 15:00:00,Group 1,Pocitos,Montevideo,France,4.0,1.0,Mexico,,4444.0,3.0,0.0,LOMBARDI Domingo (URU),CRISTOPHE Henry (BEL),REGO Gilberto (BRA),201.0,1096.0,FRA,MEX
1,1930.0,1930-07-13 15:00:00,Group 4,Parque Central,Montevideo,United States,3.0,0.0,Belgium,,18346.0,2.0,0.0,MACIAS Jose (ARG),MATEUCCI Francisco (URU),WARNKEN Alberto (CHI),201.0,1090.0,USA,BEL
2,1930.0,1930-07-14 12:45:00,Group 2,Parque Central,Montevideo,Yugoslavia,2.0,1.0,Brazil,,24059.0,2.0,0.0,TEJADA Anibal (URU),VALLARINO Ricardo (URU),BALWAY Thomas (FRA),201.0,1093.0,YUG,BRA
3,1930.0,1930-07-14 14:50:00,Group 3,Pocitos,Montevideo,Romania,3.0,1.0,Peru,,2549.0,1.0,0.0,WARNKEN Alberto (CHI),LANGENUS Jean (BEL),MATEUCCI Francisco (URU),201.0,1098.0,ROU,PER
4,1930.0,1930-07-15 16:00:00,Group 1,Parque Central,Montevideo,Argentina,1.0,0.0,France,,23409.0,0.0,0.0,REGO Gilberto (BRA),SAUCEDO Ulises (BOL),RADULESCU Constantin (ROU),201.0,1085.0,ARG,FRA


## 2. Create a team-level dataset

The original dataset has one row per match. For modeling, we need one row per team per match, so each game is represented from both team perspectives: home team and away team.

In [4]:
# Build the home-team perspective.
home_matches = matches[[
    "Year", "Stage", "Datetime", "Home Team Name", "Away Team Name",
    "Home Team Goals", "Away Team Goals", "Attendance", "MatchID"
]].copy()

home_matches = home_matches.rename(columns={
    "Home Team Name": "team",
    "Away Team Name": "opponent",
    "Home Team Goals": "goals_for",
    "Away Team Goals": "goals_against"
})
home_matches["venue_role"] = "home"

# Build the away-team perspective.
away_matches = matches[[
    "Year", "Stage", "Datetime", "Away Team Name", "Home Team Name",
    "Away Team Goals", "Home Team Goals", "Attendance", "MatchID"
]].copy()

away_matches = away_matches.rename(columns={
    "Away Team Name": "team",
    "Home Team Name": "opponent",
    "Away Team Goals": "goals_for",
    "Home Team Goals": "goals_against"
})
away_matches["venue_role"] = "away"

# Combine both perspectives into one team-match dataset.
team_matches = pd.concat([home_matches, away_matches], ignore_index=True)

team_matches["goal_difference"] = team_matches["goals_for"] - team_matches["goals_against"]
team_matches["win"] = (team_matches["goals_for"] > team_matches["goals_against"]).astype(int)
team_matches["draw"] = (team_matches["goals_for"] == team_matches["goals_against"]).astype(int)
team_matches["loss"] = (team_matches["goals_for"] < team_matches["goals_against"]).astype(int)

print("Team-match dataset shape:", team_matches.shape)
team_matches.head()

Team-match dataset shape: (1672, 14)


,Year,Stage,Datetime,team,opponent,goals_for,goals_against,Attendance,MatchID,venue_role,goal_difference,win,draw,loss
0,1930.0,Group 1,1930-07-13 15:00:00,France,Mexico,4.0,1.0,4444.0,1096.0,home,3.0,1,0,0
1,1930.0,Group 4,1930-07-13 15:00:00,United States,Belgium,3.0,0.0,18346.0,1090.0,home,3.0,1,0,0
2,1930.0,Group 2,1930-07-14 12:45:00,Yugoslavia,Brazil,2.0,1.0,24059.0,1093.0,home,1.0,1,0,0
3,1930.0,Group 3,1930-07-14 14:50:00,Romania,Peru,3.0,1.0,2549.0,1098.0,home,2.0,1,0,0
4,1930.0,Group 1,1930-07-15 16:00:00,Argentina,France,1.0,0.0,23409.0,1085.0,home,1.0,1,0,0


In [5]:
# Aggregate team performance by World Cup year.
team_tournament_stats = (
    team_matches
    .groupby(["Year", "team"], as_index=False)
    .agg(
        matches_played=("MatchID", "count"),
        goals_for=("goals_for", "sum"),
        goals_against=("goals_against", "sum"),
        wins=("win", "sum"),
        draws=("draw", "sum"),
        losses=("loss", "sum"),
        average_attendance=("Attendance", "mean")
    )
)

# Required Day 5 features.
team_tournament_stats["goals_per_match"] = (
    team_tournament_stats["goals_for"] / team_tournament_stats["matches_played"]
)
team_tournament_stats["win_percentage"] = (
    team_tournament_stats["wins"] / team_tournament_stats["matches_played"] * 100
)
team_tournament_stats["goal_difference"] = (
    team_tournament_stats["goals_for"] - team_tournament_stats["goals_against"]
)

# Round numeric columns to keep the exported file easy to read.
team_tournament_stats["goals_per_match"] = team_tournament_stats["goals_per_match"].round(2)
team_tournament_stats["win_percentage"] = team_tournament_stats["win_percentage"].round(1)
team_tournament_stats["average_attendance"] = team_tournament_stats["average_attendance"].round(0)

team_tournament_stats.head()

,Year,team,matches_played,goals_for,goals_against,wins,draws,losses,average_attendance,goals_per_match,win_percentage,goal_difference
0,1930.0,Argentina,5,18.0,9.0,4,0,1,49640.0,3.60,80.0,9.0
1,1930.0,Belgium,2,0.0,4.0,0,0,2,15173.0,0.00,0.0,-4.0
2,1930.0,Bolivia,2,0.0,8.0,0,0,2,21886.0,0.00,0.0,-8.0
3,1930.0,Brazil,2,5.0,2.0,1,0,1,24762.0,2.50,50.0,3.0
4,1930.0,Chile,3,5.0,3.0,2,0,1,17569.0,1.67,66.7,2.0


In [6]:
top_attacking_teams = duckdb.query("""
    SELECT
        Year,
        team,
        matches_played,
        goals_for,
        goals_per_match
    FROM team_tournament_stats
    WHERE matches_played >= 3
    ORDER BY goals_per_match DESC, goals_for DESC
    LIMIT 10
""").df()

top_attacking_teams

,Year,team,matches_played,goals_for,goals_per_match
0,1954.0,Hungary,5,27.0,5.40
1,1954.0,Germany,6,25.0,4.17
2,1982.0,Hungary,3,12.0,4.00
3,1958.0,France,6,23.0,3.83
4,1930.0,Uruguay,4,15.0,3.75
5,1938.0,Hungary,4,15.0,3.75
6,1950.0,Uruguay,4,15.0,3.75
7,1950.0,Brazil,6,22.0,3.67
8,1938.0,Sweden,3,11.0,3.67
9,1930.0,Argentina,5,18.0,3.60


In [7]:
best_historical_win_percentage = duckdb.query("""
    SELECT
        team,
        SUM(matches_played) AS total_matches,
        SUM(wins) AS total_wins,
        ROUND(100.0 * SUM(wins) / SUM(matches_played), 1) AS historical_win_percentage,
        SUM(goal_difference) AS historical_goal_difference
    FROM team_tournament_stats
    GROUP BY team
    HAVING SUM(matches_played) >= 10
    ORDER BY historical_win_percentage DESC, historical_goal_difference DESC
    LIMIT 10
""").df()

best_historical_win_percentage

,team,total_matches,total_wins,historical_win_percentage,historical_goal_difference
0,Brazil,104.0,70.0,67.3,119.0
1,Germany,106.0,66.0,62.3,103.0
2,Argentina,77.0,42.0,54.5,47.0
3,Italy,83.0,45.0,54.2,51.0
4,Netherlands,50.0,27.0,54.0,38.0
5,Portugal,26.0,13.0,50.0,14.0
6,Turkey,10.0,5.0,50.0,3.0
7,Denmark,16.0,8.0,50.0,3.0
8,Spain,59.0,29.0,49.2,26.0
9,Poland,31.0,15.0,48.4,4.0


In [8]:
required_feature_columns = [
    "matches_played",
    "goals_per_match",
    "win_percentage",
    "goal_difference"
]

# Confirm that all required feature columns were created.
missing_feature_columns = [
    column for column in required_feature_columns
    if column not in team_tournament_stats.columns
]

print("Missing feature columns:", missing_feature_columns)
print("Missing values in required features:")
print(team_tournament_stats[required_feature_columns].isnull().sum())
print("Final dataset shape:", team_tournament_stats.shape)

Missing feature columns: []
Missing values in required features:
matches_played     0
goals_per_match    0
win_percentage     0
goal_difference    0
dtype: int64
Final dataset shape: (425, 12)


In [9]:
team_tournament_stats.to_csv("../data/processed/clean_data.csv", index=False)

print("Saved file: ../data/processed/clean_data.csv")

Saved file: ../data/processed/clean_data.csv


## Day 5 Notes

- `goals_per_match` measures attacking performance.
- `win_percentage` measures tournament effectiveness.
- `goal_difference` summarizes the balance between goals scored and conceded.
- DuckDB was used to run SQL queries directly on pandas DataFrames.